# PoC — Agente de Voz Motiva (STT pt-BR + extração de slots)

Desafio FIAP x Motiva · MVP Persona 1 (Agente de Inspeção de Tráfego).

Meta da Sprint 1–2: provar que **voz → dados estruturados** funciona e medir um baseline.

Etapas: (1) STT, (2) NLU (intenção + slots), (3) baseline de qualidade.

## 1. STT — transcrição (pt-BR)

Opções: Whisper API (cloud) quando online, ou `faster-whisper` (local) como fallback.
Descomente conforme o que o grupo for testar. Coloque áudios de exemplo em `data/`.

In [ ]:
# STT local (offline) — requer: pip install faster-whisper
# from faster_whisper import WhisperModel
# model = WhisperModel('small', device='cpu', compute_type='int8')
# segments, info = model.transcribe('data/exemplo_km123.wav', language='pt')
# texto = ' '.join(s.text for s in segments).strip()
# print(texto)

# Para a PoC de NLU sem áudio, usamos frases de exemplo (como se já transcritas):
texto = 'registrar veículo parado no km 123 sentido norte'
print(texto)

## 2. NLU — intenção + slots

Reaproveita o serviço do backend (stub rule_based, sem chave de API). Depois a frente
Voz/ML troca por LLM com function calling, mantendo a mesma interface.

In [ ]:
import sys
sys.path.append('../backend')  # importa o NLU do backend

from app.services import nlu

r = nlu.interpretar(texto)
print('intenção:', r.intencao)
print('slots:', r.slots)
print('confiança:', r.confianca)
print('confirmação necessária:', r.confirmacao_necessaria)
print('mensagem:', r.mensagem_confirmacao)

In [ ]:
# Bateria de frases para inspecionar o comportamento do NLU
frases = [
    'registrar acidente no km 45 sentido sul',
    'preciso de apoio, manda guincho',
    'atualizar status para em atendimento',
    'tem um objeto na pista no km 10',
    'encerrar atendimento, providências tomadas',
]
for f in frases:
    r = nlu.interpretar(f)
    print(f'{f!r:55s} -> {r.intencao:24s} slots={r.slots}')

## 3. Baseline de qualidade (esboço)

Montar um pequeno conjunto rotulado (frase → intenção/slots esperados) e medir:
- **Acurácia de intenção** do NLU.
- **WER** do STT (com `jiwer`) quando houver áudios reais transcritos.

Isto vira a métrica que o grupo acompanha ao trocar rule_based por LLM e ao ajustar o STT.

In [ ]:
# Acurácia de intenção num mini-conjunto rotulado
rotulado = [
    ('registrar veículo parado no km 123 sentido norte', 'registrar_ocorrencia'),
    ('manda apoio aqui', 'solicitar_apoio'),
    ('atualizar status para em atendimento', 'atualizar_status'),
    ('encerrar atendimento', 'encerrar_atendimento'),
]
acertos = sum(nlu.interpretar(t).intencao == esperado for t, esperado in rotulado)
print(f'acurácia de intenção: {acertos}/{len(rotulado)} = {acertos/len(rotulado):.0%}')

# WER do STT (quando houver áudios + transcrições de referência):
# pip install jiwer
# import jiwer
# print('WER:', jiwer.wer(referencia, hipotese))